# Tree Canopy Detection Evaluation
Use these cells to load a trained YOLOv8 model and generate key quality metrics (F1, precision, recall, mAP, confusion matrix) on a dedicated evaluation split so you can track model quality over time.

In [1]:
# Install runtime dependencies (idempotent)
%pip install --quiet ultralytics seaborn scikit-learn pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Configure evaluation paths and shared utilities
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from ultralytics import YOLO

sns.set_theme(context="notebook", style="whitegrid")
plt.style.use("seaborn-v0_8")

DATA_YAML = Path("Tree-Top-View-6/data.yaml")
WEIGHTS_PATH = Path("Web/models/best.pt")
IMGSZ = 640
CONF_THRES = 0.25
IOU_THRES = 0.6

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Could not find dataset config at {DATA_YAML.resolve()}")
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Could not find weights at {WEIGHTS_PATH.resolve()}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO(str(WEIGHTS_PATH))

# YOLO models expose .to() in recent Ultralytics versions; guard in case older builds omit it.
if hasattr(model, "to"):
    model.to(device)

print(f"Loaded model from {WEIGHTS_PATH} on {device.upper()} with {len(model.names)} classes.")

Loaded model from Web\models\best.pt on CUDA with 1 classes.


In [3]:
# Run evaluation and display macro + per-class metrics
val_kwargs = dict(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    conf=CONF_THRES,
    iou=IOU_THRES,
    device=device,
    half=False,
    plots=True,
    save_hybrid=False,
    save_txt=False,
    verbose=True,
 )
metrics = model.val(**val_kwargs)

box_metrics = metrics.box
scalar_summary = {
    "precision_avg": float(box_metrics.p.mean()),
    "recall_avg": float(box_metrics.r.mean()),
    "f1_avg": float(box_metrics.f1.mean()),
    "map50": float(box_metrics.map50),
    "map50_95": float(box_metrics.map),
}
display(pd.DataFrame([scalar_summary]).T.rename(columns={0: "value"}))

per_class_rows = []
for cls_idx, cls_name in model.names.items():
    per_class_rows.append({
        "class_id": cls_idx,
        "class_name": cls_name,
        "precision": float(box_metrics.p[cls_idx]),
        "recall": float(box_metrics.r[cls_idx]),
        "f1": float(box_metrics.f1[cls_idx]),
    })
per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df.sort_values("f1", ascending=False).reset_index(drop=True))

speed_info = getattr(metrics, "speed", {})
if speed_info:
    display(pd.Series(speed_info, name="speed (ms)").to_frame())

WARNING 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.223  Python-3.11.13 torch-2.5.1 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Ultralytics 8.3.223  Python-3.11.13 torch-2.5.1 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.40.2 ms, read: 4.20.5 MB/s, size: 68.4 KB)
val: Fast image access  (ping: 0.40.2 ms, read: 4.20.5 MB/s, size: 68.4 KB)
val: Scanning D:\TreeSense\Tree-Top-View-6\test\labels... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 137.2it/s 0.4s.8s
val: New cache created: D:\TreeSense\Tree-Top-View-6\test\labels.cache
val: Scanning D:\TreeSense\Tree-Top-View-6\test\labels... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 137.2it/s 0.4s.8s
val: New cache created: D:\TreeSense\Tree-Top-View-6\test\labels.cache
                 Clas

,value
precision_avg,0.496616
recall_avg,0.542576
f1_avg,0.518579
map50,0.471816
map50_95,0.200663


,class_id,class_name,precision,recall,f1
0,0,tree-top,0.496616,0.542576,0.518579


,speed (ms)
preprocess,4.681577
inference,14.766015
loss,0.024525
postprocess,26.591818


In [4]:
# Visualize the confusion matrix and derive additional diagnostics
confusion = getattr(metrics, "confusion_matrix", None)
if confusion is None or not hasattr(confusion, "matrix"):
    print("Confusion matrix not available. Make sure plots=True in model.val and rerun the previous cell.")
else:
    raw_cm = np.array(confusion.matrix, dtype=float)
    label_names = list(model.names.values()) + ["background"]
    cm_df = pd.DataFrame(raw_cm, index=label_names, columns=label_names)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt=".0f", cmap="YlGnBu")
    plt.title("YOLOv8 Confusion Matrix (test split)")
    plt.ylabel("Ground truth")
    plt.xlabel("Predictions")
    plt.tight_layout()
    plt.show()

    normalized_cm = cm_df.div(cm_df.sum(axis=1).replace(0, 1), axis=0)
    display(normalized_cm.style.format("{:.2f}").set_caption("Row-normalized confusion matrix"))

    # Derive precision/recall/F1 directly from the confusion matrix for a second opinion on class balance.
    core_cm = raw_cm[:-1, :-1]  # strip background bucket
    tp = np.diag(core_cm)
    fp = core_cm.sum(axis=0) - tp
    fn = core_cm.sum(axis=1) - tp
    support = core_cm.sum(axis=1)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)
    confusion_stats = pd.DataFrame({
        "class_name": list(model.names.values()),
        "support": support.astype(int),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })
    display(confusion_stats.sort_values(["support", "f1"], ascending=[False, False]).reset_index(drop=True))

<Figure size 800x600 with 2 Axes>

,tree-top,background
tree-top,0.43,0.57
background,1.00,0.00


,class_name,support,precision,recall,f1
0,tree-top,1189,1.0,1.0,1.0
